In [7]:
from pathlib import Path
import os, json, math, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

In [8]:
# ===== Paths =====
DATA_DIR = Path("/home/danila/networks/data")
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

# whisper transcripts
TXT_DIR = DATA_DIR / "text_whisper_large_v3"
assert TXT_DIR.exists(), f"TXT_DIR not found: {TXT_DIR}"

RUN_DIR = DATA_DIR / "runs" / "finetune_mgte_text_only_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = RUN_DIR / "best_by_val_pearson.pt"
HIST_PATH = RUN_DIR / "history.json"

# ===== Task =====
EMOTIONS = ["Admiration","Amusement","Determination","Empathic Pain","Excitement","Joy"]
NUM_TARGETS = len(EMOTIONS)
ID_WIDTH = 5

# ===== Model =====
MODEL_ID = "Alibaba-NLP/gte-base-en-v1.5" #"Alibaba-NLP/gte-multilingual-base"  # mGTE family
MAX_LEN = 128

# ===== Train =====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

BATCH_SIZE = 32
EPOCHS = 30
LR_ENCODER = 1e-4
LR_HEAD = 1e-4
WEIGHT_DECAY = 0 #0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
USE_AMP = (DEVICE == "cuda")

LOSS_FN = nn.MSELoss()

L2_NORM_CLS = False

torch.backends.cuda.matmul.allow_tf32 = True

## Trying to improve winner's result

In [9]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("Device:", DEVICE)

Device: cuda


In [10]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

missing_cols = [c for c in (["Filename"] + EMOTIONS) if c not in train_df.columns]
assert not missing_cols, f"Missing cols in train CSV: {missing_cols}"

missing_cols = [c for c in (["Filename"] + EMOTIONS) if c not in valid_df.columns]
assert not missing_cols, f"Missing cols in valid CSV: {missing_cols}"

print("Train rows:", len(train_df), "| Val rows:", len(valid_df))
train_df.head()

Train rows: 8072 | Val rows: 4588


,Filename,Admiration,Amusement,Determination,Empathic Pain,Excitement,Joy
0,00000,0.333333,0.333333,0.0,0.0,0.333333,0.0
1,00001,0.000000,0.000000,0.0,0.0,0.500000,0.0
2,00002,0.500000,0.000000,0.0,0.0,0.000000,0.0
3,00003,0.000000,0.000000,0.5,0.0,0.000000,0.0
4,00004,0.400000,0.000000,0.0,0.0,0.200000,0.4


In [11]:
def txt_path(vid: str) -> Path:
    return TXT_DIR / f"{vid}.txt"

class TextEMIDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

        # filter out the missing texts
        keep_idx = []
        missing = []
        for i, row in self.df.iterrows():
            vid = row["Filename"]
            p = txt_path(vid)
            if p.exists():
                keep_idx.append(i)
            else:
                missing.append(vid)

        self.df = self.df.iloc[keep_idx].reset_index(drop=True)
        self.missing = missing

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vid = row["Filename"]
        text = txt_path(vid).read_text(encoding="utf-8").strip()
        # sometimes whisper can return empty
        if not text:
            text = "[NO SPEECH]"
        y = np.array([row[e] for e in EMOTIONS], dtype=np.float32)
        return {"id": vid, "text": text, "y": y}

train_ds = TextEMIDataset(train_df)
valid_ds = TextEMIDataset(valid_df)

print("Train usable:", len(train_ds), "missing txt:", len(train_ds.missing))
print("Val   usable:", len(valid_ds), "missing txt:", len(valid_ds.missing))

if len(train_ds.missing) or len(valid_ds.missing):
    pd.DataFrame({
        "split": ["train"]*len(train_ds.missing) + ["val"]*len(valid_ds.missing),
        "video_id": train_ds.missing + valid_ds.missing
    }).to_csv(RUN_DIR / "missing_txt.csv", index=False)
    print("Saved missing list:", RUN_DIR / "missing_txt.csv")

Train usable: 8072 missing txt: 0
Val   usable: 4588 missing txt: 0


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
    ids = [b["id"] for b in batch]

    tok = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    return {"ids": ids, "tok": tok, "y": ys}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

print("Batches | train:", len(train_loader), "val:", len(valid_loader))

Batches | train: 253 val: 144


In [13]:
class mGTERegressor(nn.Module):
    def __init__(self, model_id: str, num_targets: int, hidden: int = 512, dropout: float = 0.2, l2_norm_cls: bool = False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id, trust_remote_code=True)
        self.l2_norm_cls = l2_norm_cls

        # gte-multilingual-base hidden_size = 768
        d = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_targets)
        )

    def forward(self, tok):
        out = self.encoder(**tok)
        cls = out.last_hidden_state[:, 0, :]
        if self.l2_norm_cls:
            cls = F.normalize(cls, p=2, dim=1)
        pred = self.head(cls)
        return pred

model = mGTERegressor(MODEL_ID, NUM_TARGETS, hidden=512, dropout=0.2, l2_norm_cls=L2_NORM_CLS).to(DEVICE)
print(model)

mGTERegressor(
  (encoder): NewModel(
    (embeddings): NewEmbeddings(
      (word_embeddings): Embedding(30528, 768, padding_idx=0)
      (rotary_emb): NTKScalingRotaryEmbedding()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): NewEncoder(
      (layer): ModuleList(
        (0-11): 12 x NewLayer(
          (attention): NewSdpaAttention(
            (qkv_proj): Linear(in_features=768, out_features=2304, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (o_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): NewGatedMLP(
            (up_gate_proj): Linear(in_features=768, out_features=6144, bias=False)
            (down_proj): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUActivation()
            (hidden_dropout): Dropout(p=0.1, inplace=False)
          )
          (attn_ln): LayerNorm((768,), eps=1e-

In [14]:
@torch.no_grad()
def pearson_per_dim(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8):
    """
    pred/target: [N,6]
    returns:
      mean_corr: float
      corrs: list[float] length 6
    """
    pred = pred.float()
    target = target.float()

    vx = pred - pred.mean(dim=0, keepdim=True)
    vy = target - target.mean(dim=0, keepdim=True)

    num = (vx * vy).sum(dim=0)
    den = torch.sqrt((vx * vx).sum(dim=0) * (vy * vy).sum(dim=0)) + eps
    corr = num / den
    return corr.mean().item(), corr.detach().cpu().tolist()

In [15]:
param_groups = [
    {"params": model.encoder.parameters(), "lr": LR_ENCODER},
    {"params": model.head.parameters(), "lr": LR_HEAD},
]

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

num_training_steps = EPOCHS * len(train_loader)
num_warmup_steps = 0 #int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Steps:", num_training_steps, "Warmup:", num_warmup_steps)

Steps: 7590 Warmup: 0


/tmp/ipykernel_172304/1797801866.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [16]:
def move_tok_to_device(tok, device):
    return {k: v.to(device, non_blocking=True) for k, v in tok.items()}

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="train", leave=False):
        tok = move_tok_to_device(batch["tok"], DEVICE)
        y = batch["y"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred = model(tok)
            loss = LOSS_FN(pred, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * y.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total_loss = 0.0
    all_p, all_y = [], []

    for batch in tqdm(loader, desc="eval", leave=False):
        tok = move_tok_to_device(batch["tok"], DEVICE)
        y = batch["y"].to(DEVICE, non_blocking=True)

        pred = model(tok)
        loss = LOSS_FN(pred, y)

        total_loss += loss.item() * y.size(0)
        all_p.append(pred.detach().cpu())
        all_y.append(y.detach().cpu())

    P = torch.cat(all_p, dim=0)
    Y = torch.cat(all_y, dim=0)

    mean_corr, corrs = pearson_per_dim(P, Y)
    return {
        "loss": total_loss / len(loader.dataset),
        "mean_pearson": mean_corr,
        "per_dim": corrs,
    }

In [12]:
history = []
best_val = -1e9
best_epoch = -1
patience = 10
bad = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader)
    va = eval_model(model, valid_loader)

    row = {
        "epoch": epoch,
        "train_loss": float(tr_loss),
        "val_loss": float(va["loss"]),
        "val_mean_pearson": float(va["mean_pearson"]),
        "val_per_dim": [float(x) for x in va["per_dim"]],
        "lr_encoder": float(optimizer.param_groups[0]["lr"]),
        "lr_head": float(optimizer.param_groups[1]["lr"]),
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={tr_loss:.4f} | val_loss={va['loss']:.4f} | "
        f"val_mean_pearson={va['mean_pearson']:.4f}"
    )
    for e, c in zip(EMOTIONS, va["per_dim"]):
        print(f"  - {e:14s}: {c:.4f}")

    # save history
    with open(HIST_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    # early stopping by Pearson
    if va["mean_pearson"] > best_val + 1e-4:
        best_val = va["mean_pearson"]
        best_epoch = epoch
        bad = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "best_val_mean_pearson": best_val,
                "config": {
                    "model_id": MODEL_ID,
                    "max_len": MAX_LEN,
                    "emotions": EMOTIONS,
                    "l2_norm_cls": L2_NORM_CLS,
                    "lr_encoder": LR_ENCODER,
                    "lr_head": LR_HEAD,
                    "batch_size": BATCH_SIZE,
                }
            },
            CKPT_PATH
        )
        print(f"✅ Saved best checkpoint: epoch={epoch}, val_mean_pearson={best_val:.4f}")
    else:
        bad += 1
        print(f"⏳ No improvement: {bad}/{patience}")

    if bad >= patience:
        print(f"🛑 Early stopping. Best epoch={best_epoch}, best val_mean_pearson={best_val:.4f}")
        break

print("Done. Best:", best_val, "at epoch", best_epoch)
print("Checkpoint:", CKPT_PATH)
print("History:", HIST_PATH)

train:   0%|          | 0/253 [00:00<?, ?it/s]

/tmp/ipykernel_171095/2763919896.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  ys = torch.tensor([b["y"] for b in batch], dtype=torch.float32)
/tmp/ipykernel_171095/1310342562.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


eval:   0%|          | 0/144 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.0401 | val_loss=0.0318 | val_mean_pearson=0.4432
  - Admiration    : 0.5267
  - Amusement     : 0.4425
  - Determination : 0.4095
  - Empathic Pain : 0.5158
  - Excitement    : 0.3964
  - Joy           : 0.3686
✅ Saved best checkpoint: epoch=1, val_mean_pearson=0.4432


train:   0%|          | 0/253 [00:00<?, ?it/s]

...

eval:   0%|          | 0/144 [00:00<?, ?it/s]

Epoch 29 | train_loss=0.0029 | val_loss=0.0281 | val_mean_pearson=0.5374
  - Admiration    : 0.6031
  - Amusement     : 0.5607
  - Determination : 0.4873
  - Empathic Pain : 0.5863
  - Excitement    : 0.5039
  - Joy           : 0.4829
⏳ No improvement: 1/10


train:   0%|          | 0/253 [00:00<?, ?it/s]

eval:   0%|          | 0/144 [00:00<?, ?it/s]

Epoch 30 | train_loss=0.0029 | val_loss=0.0281 | val_mean_pearson=0.5373
  - Admiration    : 0.6030
  - Amusement     : 0.5607
  - Determination : 0.4876
  - Empathic Pain : 0.5862
  - Excitement    : 0.5038
  - Joy           : 0.4828
⏳ No improvement: 2/10
Done. Best: 0.5372796654701233 at epoch 28
Checkpoint: /home/danila/networks/data/runs/finetune_mgte_text_only_v1/best_by_val_pearson.pt
History: /home/danila/networks/data/runs/finetune_mgte_text_only_v1/history.json


In [13]:
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)

va = eval_model(model, valid_loader)
print("Best checkpoint epoch:", ckpt["epoch"])
print("Val loss:", va["loss"])
print("Val mean Pearson:", va["mean_pearson"])
for e, c in zip(EMOTIONS, va["per_dim"]):
    print(f"{e:14s}: {c:.4f}")

eval:   0%|          | 0/144 [00:00<?, ?it/s]

Best checkpoint epoch: 28
Val loss: 0.028131788240870707
Val mean Pearson: 0.5372796654701233
Admiration    : 0.6030
Amusement     : 0.5608
Determination : 0.4880
Empathic Pain : 0.5846
Excitement    : 0.5041
Joy           : 0.4832


In [14]:
@torch.no_grad()
def predict_split(model, loader):
    model.eval()
    out = []
    for batch in tqdm(loader, desc="predict", leave=False):
        tok = move_tok_to_device(batch["tok"], DEVICE)
        pred = model(tok).detach().cpu().numpy()
        for vid, p in zip(batch["ids"], pred):
            out.append({"Filename": vid, **{k: float(v) for k, v in zip(EMOTIONS, p)}})
    return pd.DataFrame(out)

pred_val_df = predict_split(model, valid_loader)
pred_val_df.head()

predict:   0%|          | 0/144 [00:00<?, ?it/s]

,Filename,Admiration,Amusement,Determination,Empathic Pain,Excitement,Joy
0,08072,0.059690,0.152923,0.021162,0.027916,0.352601,0.005670
1,08073,0.162595,0.111621,0.100409,0.021700,0.117232,0.110943
2,08074,0.231706,0.004219,0.022610,-0.001488,0.520084,0.005527
3,08075,-0.005004,-0.000545,0.193217,0.058282,0.083596,0.004215
4,08076,0.026377,0.021618,0.191718,0.058089,0.318281,0.071086


In [17]:
# === Save best fine-tuned encoder for embeddings (HF format) ===
from pathlib import Path
import json
import torch

ENCODER_SAVE_DIR = RUN_DIR / "best_encoder_for_embeddings"
ENCODER_SAVE_DIR.mkdir(parents=True, exist_ok=True)

assert CKPT_PATH.exists(), f"Best checkpoint not found: {CKPT_PATH}"

# Load best checkpoint and restore weights into model
ckpt = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.eval()

# Save ONLY encoder weights in HuggingFace format
# safetensors if available via transformers (safe_serialization=True)
model.encoder.save_pretrained(ENCODER_SAVE_DIR, safe_serialization=True)

# Save tokenizer
tokenizer.save_pretrained(ENCODER_SAVE_DIR)

# Save pooling/embedding recipe metadata
embed_meta = {
    "source_ckpt": str(CKPT_PATH),
    "best_epoch": int(ckpt.get("epoch", -1)),
    "best_val_mean_pearson": float(ckpt.get("best_val_mean_pearson", -1.0)),
    "base_model_id": MODEL_ID,
    "max_length": int(MAX_LEN),
    "embedding_pooling": "cls",          # out.last_hidden_state[:,0]
    "l2_norm_cls": bool(L2_NORM_CLS),
}
with open(ENCODER_SAVE_DIR / "embedding_recipe.json", "w", encoding="utf-8") as f:
    json.dump(embed_meta, f, ensure_ascii=False, indent=2)

print("✅ Saved fine-tuned encoder for embeddings to:", ENCODER_SAVE_DIR)
print("   Files:", [p.name for p in sorted(ENCODER_SAVE_DIR.iterdir())])

✅ Saved fine-tuned encoder for embeddings to: /home/danila/networks/data/runs/finetune_mgte_text_only_v1/best_encoder_for_embeddings
   Files: ['config.json', 'configuration.py', 'embedding_recipe.json', 'model.safetensors', 'modeling.py', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']


# Recalculating embeggings with finetuend encoder

In [20]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ===== Where finetuned encoder is saved =====
ENC_SAVE_DIR = Path("/home/danila/networks/data/runs/finetune_mgte_text_only_v1/best_encoder_for_embeddings")
assert ENC_SAVE_DIR.exists(), f"Finetuned encoder dir not found: {ENC_SAVE_DIR}"

# ===== Input texts =====
TXT_DIR = DATA_DIR / "text_whisper_large_v3"
assert TXT_DIR.exists(), f"TXT_DIR not found: {TXT_DIR}"

# ===== Which ids to embed =====
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"
ID_WIDTH = 5

# ===== Output embeddings dir =====
OUT_EMB_DIR = DATA_DIR / "embeddings" / "text_gte_base_en_v1_5_finetuned_cls128"
OUT_EMB_DIR.mkdir(parents=True, exist_ok=True)

# ===== Embedding params =====
MAX_LEN = 128
BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

L2_NORM_CLS = False

print("ENC_SAVE_DIR:", ENC_SAVE_DIR)
print("OUT_EMB_DIR :", OUT_EMB_DIR)
print("DEVICE      :", DEVICE)

ENC_SAVE_DIR: /home/danila/networks/data/runs/finetune_mgte_text_only_v1/best_encoder_for_embeddings
OUT_EMB_DIR : /home/danila/networks/data/embeddings/text_gte_base_en_v1_5_finetuned_cls128
DEVICE      : cuda


In [21]:
tokenizer_ft = AutoTokenizer.from_pretrained(ENC_SAVE_DIR, use_fast=True)
encoder_ft = AutoModel.from_pretrained(ENC_SAVE_DIR, trust_remote_code=True).to(DEVICE).eval()

print("Loaded finetuned encoder:", ENC_SAVE_DIR)
print("Hidden size:", encoder_ft.config.hidden_size)

Loaded finetuned encoder: /home/danila/networks/data/runs/finetune_mgte_text_only_v1/best_encoder_for_embeddings
Hidden size: 768


In [22]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

all_ids = sorted(set(train_df["Filename"].tolist()) | set(valid_df["Filename"].tolist()))
print("Total ids:", len(all_ids))

def txt_path(vid: str) -> Path:
    return TXT_DIR / f"{vid}.txt"

missing_txt = [vid for vid in all_ids if not txt_path(vid).exists()]
print("Missing txt:", len(missing_txt))
if len(missing_txt) and len(missing_txt) < 30:
    print("Examples:", missing_txt[:30])

Total ids: 12660
Missing txt: 0


In [23]:
@torch.inference_mode()
def embed_text_batch(texts, max_len=128, l2_norm=False):
    batch = tokenizer_ft(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

    out = encoder_ft(**batch)
    cls = out.last_hidden_state[:, 0, :]  # CLS

    if l2_norm:
        cls = F.normalize(cls, p=2, dim=1)

    return cls.detach().float().cpu().numpy()  # [B, D]

In [24]:
import time

def read_text_or_stub(vid: str) -> str:
    p = txt_path(vid)
    if not p.exists():
        return None
    t = p.read_text(encoding="utf-8").strip()
    return t if t else "[NO SPEECH]"

# skip already computed
existing = set([p.stem for p in OUT_EMB_DIR.glob("*.npz")])
print("Already computed:", len(existing))

todo_ids = [vid for vid in all_ids if vid not in existing and txt_path(vid).exists()]
print("To compute:", len(todo_ids))

meta_rows = []
t0 = time.time()

for i in tqdm(range(0, len(todo_ids), BATCH_SIZE), desc="Embed batches"):
    batch_ids = todo_ids[i:i+BATCH_SIZE]
    texts = [read_text_or_stub(vid) for vid in batch_ids]

    # safety: filter any None
    keep = [(vid, txt) for vid, txt in zip(batch_ids, texts) if txt is not None]
    if not keep:
        continue

    batch_ids2, texts2 = zip(*keep)
    embs = embed_text_batch(list(texts2), max_len=MAX_LEN, l2_norm=L2_NORM_CLS)

    for vid, emb in zip(batch_ids2, embs):
        out_path = OUT_EMB_DIR / f"{vid}.npz"
        np.savez_compressed(out_path, embedding=emb.astype(np.float32))

        meta_rows.append({
            "video_id": vid,
            "txt_chars": int(len(txt_path(vid).read_text(encoding="utf-8"))),
            "emb_dim": int(emb.shape[0]),
            "path": str(out_path),
        })

dt = time.time() - t0
print(f"Done in {dt:.1f} sec. Saved new:", len(meta_rows))

Already computed: 0
To compute: 12660


Embed batches:   0%|          | 0/50 [00:00<?, ?it/s]

Done in 22.9 sec. Saved new: 12660


In [25]:
meta_df = pd.DataFrame(meta_rows)
index_path = OUT_EMB_DIR / "embeddings_index.parquet"
meta_df.to_parquet(index_path, index=False)

run_meta = {
    "encoder_dir": str(ENC_SAVE_DIR),
    "txt_dir": str(TXT_DIR),
    "out_emb_dir": str(OUT_EMB_DIR),
    "max_len": int(MAX_LEN),
    "batch_size": int(BATCH_SIZE),
    "l2_norm_cls": bool(L2_NORM_CLS),
    "num_embedded_new": int(len(meta_df)),
}

(OUT_EMB_DIR / "run_meta.json").write_text(
    json.dumps(run_meta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("✅ Saved index:", index_path)
print("✅ Saved meta :", OUT_EMB_DIR / "run_meta.json")
meta_df.head()

✅ Saved index: /home/danila/networks/data/embeddings/text_gte_base_en_v1_5_finetuned_cls128/embeddings_index.parquet
✅ Saved meta : /home/danila/networks/data/embeddings/text_gte_base_en_v1_5_finetuned_cls128/run_meta.json


,video_id,txt_chars,emb_dim,path
0,00000,60,768,/home/danila/networks/data/embeddings/text_gte...
1,00001,45,768,/home/danila/networks/data/embeddings/text_gte...
2,00002,134,768,/home/danila/networks/data/embeddings/text_gte...
3,00003,98,768,/home/danila/networks/data/embeddings/text_gte...
4,00004,61,768,/home/danila/networks/data/embeddings/text_gte...


In [26]:
# sanity check
a = "I am so happy today!"
b = "This is wonderful and exciting!"
c = "I am devastated and in pain."

Ea = embed_text_batch([a], max_len=MAX_LEN, l2_norm=True)[0]
Eb = embed_text_batch([b], max_len=MAX_LEN, l2_norm=True)[0]
Ec = embed_text_batch([c], max_len=MAX_LEN, l2_norm=True)[0]

def cos(x,y):
    return float(np.dot(x,y) / (np.linalg.norm(x)*np.linalg.norm(y) + 1e-9))

print("cos(a,b) =", cos(Ea,Eb))
print("cos(a,c) =", cos(Ea,Ec))

cos(a,b) = 0.713760256767273
cos(a,c) = 0.7833521962165833
